In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
import torch.autograd as autograd
from torch.utils.data import Dataset, DataLoader

from einops import rearrange, reduce, repeat
from einops.layers.torch import Rearrange, Reduce

# Add parent directory to path for imports to find losses.py
# Assuming notebook is in test_unimib_new/ folder
%cd ..
try:
    from losses import TimeSeriesLoss
    print("Successfully imported TimeSeriesLoss from losses.py")
except ImportError:
    print("Could not import losses.py. Make sure you are in the correct directory.")
%cd test_unimib_running
print(f"Current working directory: {os.getcwd()}")

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Select losses to run
LOSSES_TO_RUN = ["tmw","opw","taot","sdtw","awswd","tcot","pow","gow"]
EVAL_ONLY = False # Set to True to skip training and just evaluate existing checkpoints

# Training configuration
TRAIN_CONFIG = {
    "epochs": 3000,
    "batch_size": 128,
    "g_lr": 3e-4,
    "c_lr": 1e-3,
    "latent_dim": 100,
    "seq_len": 150,
    "channels": 3,  
    "patch_size": 30,  
    "n_critic": 3,
    "lambda_gp": 10,
    "lambda_ts": 0.05,
    "lambda_sm": 1,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "save_interval": 50,
    "num_eval_samples": 1000,
    "output_dir": "."
}

# Per-loss configuration overrides
# Use this dictionary to override TRAIN_CONFIG values for specific losses
LOSS_CONFIG_OVERRIDES = {
    "opw": {
        "lambda_ts": 0.1
    },
    "taot": {
        "lambda_ts": 0.1
    },
    "tcot": {
        "lambda_ts": 0.1
    },
    "awswd": {
        "lambda_ts": 0.1
    },
    "tmw": {
        "lambda_ts": 0.5
    },
    "sdtw": {
        "lambda_ts": 0.1
    },
    "pow": {
        "lambda_ts": 0.1
    },
    "gow": {
        "lambda_ts": 0.1
    }
}

# Loss-specific hyperparameters
LOSS_HYPERPARAMS = {                                                
    "opw": {
        "lambda1": 20,
        "lambda2": 1.0,
        "sigma": 1.0,
        "num_iter": 20
    },
    "taot": {
        "reg_lambda": 50.0,
        "time_weight": 1.0,
        "num_iter": 100
    },
    "tcot": {
        "reg_lambda": 10.0,
        "num_iter": 100
    },
    "awswd": {
        "reg_lambda": 50.0,
        "l_window": 5,
        "k_steep": 0.1,
        "num_sinkhorn": 50,
        "num_outer": 5
    },
    "tmw": {
        "cost_function": "L2",
        "mask_type": 2,
        "reg": 1,
        "max_iterations": 50,
        "thres": 1e-5,
        "eps_threshold": 0.1,
        "masked": True,
        "rescale": True,
        "device": TRAIN_CONFIG['device']
    },
    "sdtw": {
        "use_cuda": True,
        "gamma": 0.1,
        "normalize": False,                                                                                                                                                                                                                                                                                                                                                                                             
        "bandwidth": None
    },
    "pow": {'order_reg': 1.0, 'sinkhorn_reg': 0.1, 'm_mass': 0.9, 'num_iter': 20},
    'gow': {'lambda1': 5.0, 'lambda2': 0.1, 'max_iter': 5, 'sinkhorn_iter': 20, 'fw_iter': 10}
}

print(f"Device: {TRAIN_CONFIG['device']}")

In [ ]:
# ============================================================================
# MODEL DEFINITIONS
# ============================================================================

class MultiHeadAttention(nn.Module):
    def __init__(self, emb_size, num_heads, dropout):
        super().__init__()
        self.emb_size = emb_size
        self.num_heads = num_heads
        self.keys = nn.Linear(emb_size, emb_size)
        self.queries = nn.Linear(emb_size, emb_size)
        self.values = nn.Linear(emb_size, emb_size)
        self.att_drop = nn.Dropout(dropout)
        self.projection = nn.Linear(emb_size, emb_size)

    def forward(self, x: Tensor, mask: Tensor = None) -> Tensor:
        queries = rearrange(self.queries(x), "b n (h d) -> b h n d", h=self.num_heads)
        keys = rearrange(self.keys(x), "b n (h d) -> b h n d", h=self.num_heads)
        values = rearrange(self.values(x), "b n (h d) -> b h n d", h=self.num_heads)
        energy = torch.einsum('bhqd, bhkd -> bhqk', queries, keys)
        if mask is not None:
            fill_value = torch.finfo(torch.float32).min
            energy.mask_fill(~mask, fill_value)
        scaling = self.emb_size ** (1 / 2)
        att = F.softmax(energy / scaling, dim=-1)
        att = self.att_drop(att)
        out = torch.einsum('bhal, bhlv -> bhav ', att, values)
        out = rearrange(out, "b h n d -> b n (h d)")
        out = self.projection(out)
        return out


class ResidualAdd(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x, **kwargs):
        res = x
        x = self.fn(x, **kwargs)
        x += res
        return x


class FeedForwardBlock(nn.Sequential):
    def __init__(self, emb_size, expansion, drop_p):
        super().__init__(
            nn.Linear(emb_size, expansion * emb_size),
            nn.GELU(),
            nn.Dropout(drop_p),
            nn.Linear(expansion * emb_size, emb_size),
        )


class Gen_TransformerEncoderBlock(nn.Sequential):
    def __init__(self, emb_size, num_heads=5, drop_p=0.5, forward_expansion=4, forward_drop_p=0.5):
        super().__init__(
            ResidualAdd(nn.Sequential(
                nn.LayerNorm(emb_size),
                MultiHeadAttention(emb_size, num_heads, drop_p),
                nn.Dropout(drop_p)
            )),
            ResidualAdd(nn.Sequential(
                nn.LayerNorm(emb_size),
                FeedForwardBlock(emb_size, expansion=forward_expansion, drop_p=forward_drop_p),
                nn.Dropout(drop_p)
            )))


class Gen_TransformerEncoder(nn.Sequential):
    def __init__(self, depth=8, **kwargs):
        super().__init__(*[Gen_TransformerEncoderBlock(**kwargs) for _ in range(depth)])


class Generator(nn.Module):
    def __init__(self, seq_len=150, patch_size=15, channels=3, num_classes=9, latent_dim=100, 
                 embed_dim=10, depth=3, num_heads=5, forward_drop_rate=0.5, attn_drop_rate=0.5):
        super(Generator, self).__init__()
        self.channels = channels
        self.latent_dim = latent_dim
        self.seq_len = seq_len
        self.embed_dim = embed_dim
        self.patch_size = patch_size
        self.depth = depth
        self.attn_drop_rate = attn_drop_rate
        self.forward_drop_rate = forward_drop_rate

        self.l1 = nn.Linear(self.latent_dim, self.seq_len * self.embed_dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.seq_len, self.embed_dim))
        self.blocks = Gen_TransformerEncoder(
            depth=self.depth,
            emb_size=self.embed_dim,
            drop_p=self.attn_drop_rate,
            forward_drop_p=self.forward_drop_rate
        )
        self.deconv = nn.Sequential(nn.Conv2d(self.embed_dim, self.channels, 1, 1, 0))

    def forward(self, z):
        x = self.l1(z).view(-1, self.seq_len, self.embed_dim)
        x = x + self.pos_embed
        H, W = 1, self.seq_len
        x = self.blocks(x)
        x = x.reshape(x.shape[0], 1, x.shape[1], x.shape[2])
        output = self.deconv(x.permute(0, 3, 1, 2))
        output = output.view(-1, self.channels, H, W)
        return output


class Dis_TransformerEncoderBlock(nn.Sequential):
    def __init__(self, emb_size=100, num_heads=5, drop_p=0., forward_expansion=4, forward_drop_p=0.):
        super().__init__(
            ResidualAdd(nn.Sequential(
                nn.LayerNorm(emb_size),
                MultiHeadAttention(emb_size, num_heads, drop_p),
                nn.Dropout(drop_p)
            )),
            ResidualAdd(nn.Sequential(
                nn.LayerNorm(emb_size),
                FeedForwardBlock(emb_size, expansion=forward_expansion, drop_p=forward_drop_p),
                nn.Dropout(drop_p)
            )))


class Dis_TransformerEncoder(nn.Sequential):
    def __init__(self, depth=8, **kwargs):
        super().__init__(*[Dis_TransformerEncoderBlock(**kwargs) for _ in range(depth)])


class ClassificationHead(nn.Sequential):
    def __init__(self, emb_size=100, n_classes=2):
        super().__init__()
        self.clshead = nn.Sequential(
            Reduce('b n e -> b e', reduction='mean'),
            nn.LayerNorm(emb_size),
            nn.Linear(emb_size, n_classes)
        )

    def forward(self, x):
        return self.clshead(x)


class PatchEmbedding_Linear(nn.Module):
    def __init__(self, in_channels=21, patch_size=16, emb_size=100, seq_length=1024):
        super().__init__()
        self.projection = nn.Sequential(
            Rearrange('b c h (w s2) -> b w (h s2 c)', s2=patch_size),
            nn.Linear(patch_size * in_channels, emb_size)
        )
        self.cls_token = nn.Parameter(torch.randn(1, 1, emb_size))
        self.positions = nn.Parameter(torch.randn((seq_length // patch_size) + 1, emb_size))

    def forward(self, x: Tensor) -> Tensor:
        b, _, _, _ = x.shape
        x = self.projection(x)
        cls_tokens = repeat(self.cls_token, '() n e -> b n e', b=b)
        x = torch.cat([cls_tokens, x], dim=1)
        x += self.positions
        return x


class Discriminator(nn.Sequential):
    def __init__(self, in_channels=3, patch_size=15, emb_size=50, seq_length=150, depth=3, n_classes=1, **kwargs):
        super().__init__(
            PatchEmbedding_Linear(in_channels, patch_size, emb_size, seq_length),
            Dis_TransformerEncoder(depth, emb_size=emb_size, drop_p=0.5, forward_drop_p=0.5, **kwargs),
            ClassificationHead(emb_size, n_classes)
        )

In [ ]:
# ============================================================================
# DATASET
# ============================================================================

class TimeSeriesDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


import scipy.io

# Define the data type and path
data_type = 'adl'
data_path = '../consolidated_datasets/UniMiB-SHAR/data/'

# Load the .mat files
data_dict = scipy.io.loadmat(f'{data_path}{data_type}_data.mat')
labels_dict = scipy.io.loadmat(f'{data_path}{data_type}_labels.mat')
names_dict = scipy.io.loadmat(f'{data_path}{data_type}_names.mat')

# Extract the data from the dictionaries
# The actual variable names inside the .mat files might vary,
# but they are likely to be similar to the file names.
# You can inspect the keys of the loaded dictionaries to be sure.
# For example: print(data_dict.keys())
acc_data = data_dict[f'{data_type}_data']
labels = labels_dict[f'{data_type}_labels']
activity_names = names_dict[f'{data_type}_names']



clean_activity_names = [name[0][0] for name in activity_names]

# Find the index of 'running'. We add 1 because MATLAB labels are 1-based.
try:
    running_label = clean_activity_names.index('running') + 1
except ValueError:
    running_label = -1
# -------------------------------------------------------------------
# Step 2: Filter the accelerometer data using the running label
# -------------------------------------------------------------------

if running_label != -1:
    # Get the first column of the labels array, which contains the activity IDs.
    activity_ids = labels[:, 0]

    # Find the indices of all rows where the activity ID matches the running label.
    running_indices = np.where(activity_ids == running_label)[0]

    # Use these indices to select the corresponding data from acc_data.
    running_data = acc_data[running_indices, :]

#reshape from (num_sample,453) to (num_sample,3,1,151)    
running_data = running_data.reshape(-1, 3, 1, 151)
running_data = running_data[:,:,:,:150]

from sklearn.preprocessing import MinMaxScaler
normalizer = MinMaxScaler(feature_range=(-1, 1))
running_data = normalizer.fit_transform(running_data.reshape(-1, 150)).reshape(running_data.shape)
running_data = torch.tensor(running_data).float()

def load_dataset(running_data):
    return TimeSeriesDataset(running_data)

# Load Data
try:
    dataset = load_dataset(running_data)
    dataloader = DataLoader(dataset, batch_size=TRAIN_CONFIG['batch_size'], shuffle=True)
    print(f"Dataset loaded. Size: {len(dataset)}")
except Exception as e:
    print(f"Error loading dataset: {e}")

In [ ]:
# ============================================================================
# TRAINING UTILITIES
# ============================================================================

def compute_gradient_penalty(critic, real_samples, fake_samples, device):
    """Calculates the gradient penalty loss for WGAN GP."""
    alpha = torch.rand(real_samples.size(0), 1, 1, 1, device=device)
    interpolates = (alpha * real_samples + ((1 - alpha) * fake_samples)).requires_grad_(True)
    critic_interpolates = critic(interpolates)
    gradients = autograd.grad(
        outputs=critic_interpolates,
        inputs=interpolates,
        grad_outputs=torch.ones_like(critic_interpolates),
        create_graph=True,
        retain_graph=True,
    )[0]
    gradients = gradients.reshape(gradients.size(0), -1)
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gradient_penalty


def save_checkpoint(ckpt_dir, generator, critic, optimizer_G, optimizer_C, epoch):
    """Save checkpoint, removing old ones to save space."""
    # Remove old checkpoints
    for f in os.listdir(ckpt_dir):
        if f.startswith("checkpoint_") and f.endswith(".pth"):
            os.remove(os.path.join(ckpt_dir, f))
    
    # Save new checkpoint
    checkpoint = {
        'epoch': epoch,
        'generator_state_dict': generator.state_dict(),
        'critic_state_dict': critic.state_dict(),
        'optimizer_G_state_dict': optimizer_G.state_dict(),
        'optimizer_C_state_dict': optimizer_C.state_dict(),
    }
    torch.save(checkpoint, os.path.join(ckpt_dir, f"checkpoint_latest.pth"))
    print(f"  [Checkpoint] Saved at epoch {epoch}")


def load_checkpoint(ckpt_dir, generator, critic, optimizer_G, optimizer_C, device):
    """Load checkpoint if exists."""
    ckpt_path = os.path.join(ckpt_dir, "checkpoint_latest.pth")
    if os.path.exists(ckpt_path):
        checkpoint = torch.load(ckpt_path, map_location=device)
        generator.load_state_dict(checkpoint['generator_state_dict'])
        critic.load_state_dict(checkpoint['critic_state_dict'])
        optimizer_G.load_state_dict(checkpoint['optimizer_G_state_dict'])
        optimizer_C.load_state_dict(checkpoint['optimizer_C_state_dict'])
        start_epoch = checkpoint['epoch']
        print(f"  [Checkpoint] Resumed from epoch {start_epoch}")
        return start_epoch
    return 0

In [ ]:
# ============================================================================
# TRAINING FUNCTION
# ============================================================================

def train_single_loss(loss_name, config, loss_params, dataloader, output_dir):
    """Train the GAN with a single loss function."""
    print(f"\n{'='*60}")
    print(f"Training with loss: {loss_name.upper()}")
    print(f"{'='*60}")
    
    device = config['device']
    ckpt_dir = os.path.join(output_dir, f"ckpt_{loss_name}")
    os.makedirs(ckpt_dir, exist_ok=True)

    # Initialize models
    generator = Generator(
        seq_len=config['seq_len'],
        patch_size=config['patch_size'],
        channels=config['channels'],
        latent_dim=config['latent_dim'],
        depth=5
    ).to(device)

    critic = Discriminator(
        in_channels=config['channels'],
        patch_size=config['patch_size'],
        seq_length=config['seq_len'],
        n_classes=1
    ).to(device)

    # Initialize loss
    ts_loss = TimeSeriesLoss(loss_name, loss_params)
    smooth_loss = nn.L1Loss()

    # Optimizers
    optimizer_G = torch.optim.Adam(generator.parameters(), lr=config['g_lr'], betas=(0.0, 0.9))
    optimizer_C = torch.optim.Adam(critic.parameters(), lr=config['c_lr'], betas=(0.0, 0.9))

    # Load checkpoint if exists
    start_epoch = load_checkpoint(ckpt_dir, generator, critic, optimizer_G, optimizer_C, device)

    # Training loop
    critic_losses = []
    generator_losses = []

    for epoch in range(start_epoch, config['epochs']):
        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{config['epochs']}", leave=False)

        for real_series in progress_bar:
            real_series = real_series.to(device)

            # Train Critic
            for _ in range(config['n_critic']):
                optimizer_C.zero_grad()
                noise = torch.randn(real_series.size(0), config['latent_dim'], device=device)
                fake_series = generator(noise).detach()

                real_scores = critic(real_series)
                fake_scores = critic(fake_series)

                loss_w = torch.mean(fake_scores) - torch.mean(real_scores)
                gradient_penalty = compute_gradient_penalty(critic, real_series, fake_series, device)
                critic_loss = loss_w + config['lambda_gp'] * gradient_penalty
                critic_loss.backward()
                optimizer_C.step()

            # Train Generator
            optimizer_G.zero_grad()
            noise = torch.randn(real_series.size(0), config['latent_dim'], device=device)
            fake_series_for_g = generator(noise)

            adversarial_loss = -torch.mean(critic(fake_series_for_g))
            
            # Time-Series Loss (reshape to batch, seq_len, dim)
            fake_reshaped = fake_series_for_g.squeeze(2).permute(0, 2, 1)
            real_reshaped = real_series.squeeze(2).permute(0, 2, 1)
            
            ts_loss_val = ts_loss(fake_reshaped, real_reshaped)
            if isinstance(ts_loss_val, tuple):
                ts_loss_val = ts_loss_val[0]
            ts_loss_mean = ts_loss_val.mean() if ts_loss_val.dim() > 0 else ts_loss_val

            l2_smooth_loss = smooth_loss(fake_series_for_g[:, :, :, 1:], fake_series_for_g[:, :, :, :-1])

            generator_loss = adversarial_loss + config['lambda_ts'] * ts_loss_mean + config['lambda_sm'] * l2_smooth_loss
            generator_loss.backward()
            optimizer_G.step()

            progress_bar.set_postfix({
                "C_Loss": f"{critic_loss.item():.4f}",
                "G_Loss": f"{generator_loss.item():.4f}"
            })

        critic_losses.append(critic_loss.item())
        generator_losses.append(generator_loss.item())

        # Save checkpoint
        if (epoch + 1) % config['save_interval'] == 0:
            save_checkpoint(ckpt_dir, generator, critic, optimizer_G, optimizer_C, epoch + 1)

    # Save final model
    save_checkpoint(ckpt_dir, generator, critic, optimizer_G, optimizer_C, config['epochs'])
    torch.save(generator.state_dict(), os.path.join(ckpt_dir, "generator_final.pth"))

    # Save loss plot
    plt.figure(figsize=(10, 5))
    plt.title(f"Training Losses - {loss_name.upper()}")
    plt.plot(generator_losses, label="Generator")
    plt.plot(critic_losses, label="Critic")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, f"{loss_name}_training_loss.png"), dpi=150)
    plt.show()

    print(f"Training completed for {loss_name.upper()}")
    return generator

In [ ]:
# ============================================================================
# EVALUATION FUNCTIONS
# ============================================================================

def generate_samples(generator, num_samples, latent_dim, device, seq_len, channels):
    """Generate fake samples from the generator."""
    generator.eval()
    fake_samples = []
    with torch.no_grad():
        for _ in range(num_samples):
            noise = torch.randn(1, latent_dim, device=device)
            fake_sample = generator(noise).cpu()
            fake_samples.append(fake_sample)
    fake_samples = torch.cat(fake_samples, dim=0)
    return fake_samples.squeeze().reshape(num_samples, seq_len, channels).numpy()


def feature_extract(dist_fake, dist_real):
    """Extract features using MiniRocket."""
    from sktime.transformations.panel.rocket import MiniRocketMultivariate
    dist = np.concatenate((dist_fake, dist_real), axis=0)
    dist = dist.transpose(0, 2, 1)
    rocket = MiniRocketMultivariate()
    features = rocket.fit_transform(dist)
    features_fake = features[:len(dist_fake)]
    features_real = features[len(dist_fake):]
    return features_fake, features_real


def compute_mmd(features_fake, features_real, device='cpu'):
    """Compute Maximum Mean Discrepancy."""
    dist_fake = torch.tensor(features_fake.to_numpy() if hasattr(features_fake, 'to_numpy') else features_fake, 
                             dtype=torch.float32).to(device)
    dist_real = torch.tensor(features_real.to_numpy() if hasattr(features_real, 'to_numpy') else features_real, 
                             dtype=torch.float32).to(device)

    xx = torch.matmul(dist_fake, dist_fake.t())
    yy = torch.matmul(dist_real, dist_real.t())
    zz = torch.matmul(dist_fake, dist_real.t())
    
    rx = xx.diag().unsqueeze(0).expand_as(xx)
    ry = yy.diag().unsqueeze(0).expand_as(yy)

    dxx = rx.t() + rx - 2. * xx
    dyy = ry.t() + ry - 2. * yy
    dxy = rx.t() + ry - 2. * zz

    all_distances = torch.cat([dxx.flatten(), dyy.flatten(), dxy.flatten()])
    median_sq_dist = torch.median(all_distances).clamp(min=1e-6)
    sigma = torch.sqrt(median_sq_dist / 2.0)

    XX = torch.exp(-dxx / (2.0 * sigma**2))
    YY = torch.exp(-dyy / (2.0 * sigma**2))
    XY = torch.exp(-dxy / (2.0 * sigma**2))

    return (torch.mean(XX) + torch.mean(YY) - 2.0 * torch.mean(XY)).item()


def matrix_sqrt(matrix):
    """
    Compute the matrix square root using eigendecomposition.
    For a symmetric positive semi-definite matrix A, sqrt(A) = V @ sqrt(D) @ V^T
    where A = V @ D @ V^T is the eigendecomposition.
    """
    # Ensure symmetry
    matrix = (matrix + matrix.T) / 2
    
    # Eigendecomposition
    eigenvalues, eigenvectors = torch.linalg.eigh(matrix)
    
    # Clamp eigenvalues to avoid numerical issues with negative values
    eigenvalues = torch.clamp(eigenvalues, min=0)
    
    # Compute sqrt of eigenvalues
    sqrt_eigenvalues = torch.sqrt(eigenvalues)
    
    # Reconstruct matrix sqrt: V @ sqrt(D) @ V^T
    sqrt_matrix = eigenvectors @ torch.diag(sqrt_eigenvalues) @ eigenvectors.T
    
    return sqrt_matrix


def compute_frechet_distance(features_fake, features_real, device='cpu'):
    """
    Compute Frechet Distance (FID-like) using ROCKET features with PyTorch.
    FD = ||mu_r - mu_g||^2 + Tr(C_r + C_g - 2*sqrt(C_r * C_g))
    """
    # Convert to torch tensors
    if hasattr(features_fake, 'to_numpy'):
        feat_fake = torch.tensor(features_fake.to_numpy(), dtype=torch.float64, device=device)
    else:
        feat_fake = torch.tensor(features_fake, dtype=torch.float64, device=device)
    
    if hasattr(features_real, 'to_numpy'):
        feat_real = torch.tensor(features_real.to_numpy(), dtype=torch.float64, device=device)
    else:
        feat_real = torch.tensor(features_real, dtype=torch.float64, device=device)
    
    # Compute mean
    mu_real = torch.mean(feat_real, dim=0)
    mu_fake = torch.mean(feat_fake, dim=0)
    
    # Compute covariance (unbiased estimator)
    # cov(X) = (X - mean)^T @ (X - mean) / (N - 1)
    feat_real_centered = feat_real - mu_real
    feat_fake_centered = feat_fake - mu_fake
    
    n_real = feat_real.shape[0]
    n_fake = feat_fake.shape[0]
    
    sigma_real = (feat_real_centered.T @ feat_real_centered) / (n_real - 1)
    sigma_fake = (feat_fake_centered.T @ feat_fake_centered) / (n_fake - 1)
    
    # Compute FD
    diff = mu_real - mu_fake
    
    # Compute sqrt(sigma_real @ sigma_fake)
    # For numerical stability, add small regularization if needed
    try:
        product = sigma_real @ sigma_fake
        covmean = matrix_sqrt(product)
    except Exception:
        # Add regularization if matrix sqrt fails
        offset = torch.eye(sigma_real.shape[0], dtype=torch.float64, device=device) * 1e-6
        product = (sigma_real + offset) @ (sigma_fake + offset)
        covmean = matrix_sqrt(product)
    
    # FD = ||mu_r - mu_g||^2 + Tr(C_r) + Tr(C_g) - 2*Tr(sqrt(C_r @ C_g))
    fd = torch.dot(diff, diff) + torch.trace(sigma_real) + torch.trace(sigma_fake) - 2 * torch.trace(covmean)
    
    return float(fd.item())


# ============================================================================
# ACS and AJSD Metrics (TTS-GAN protocol)
# ============================================================================

def extract_statistical_features(data):
    """
    Extract 7 statistical features per channel: Mean, Variance, Skewness, Kurtosis, Min, Max, Median.
    Input: data of shape (N, seq_len, channels)
    Output: features of shape (N, 7 * channels)
    """
    from scipy.stats import skew, kurtosis
    
    N, T, C = data.shape
    features = np.zeros((N, 7 * C))
    
    for i in range(N):
        for c in range(C):
            seq = data[i, :, c]
            features[i, c * 7 + 0] = np.mean(seq)
            features[i, c * 7 + 1] = np.var(seq)
            features[i, c * 7 + 2] = skew(seq)
            features[i, c * 7 + 3] = kurtosis(seq)
            features[i, c * 7 + 4] = np.min(seq)
            features[i, c * 7 + 5] = np.max(seq)
            features[i, c * 7 + 6] = np.median(seq)
    
    return features


def compute_acs(real_data, fake_data):
    """
    Compute Average Cosine Similarity (ACS) between real and fake data.
    Score in [-1, 1], closer to 1 is better.
    """
    feat_real = extract_statistical_features(real_data)
    feat_fake = extract_statistical_features(fake_data)
    
    # Compute pairwise cosine similarity
    N = min(len(feat_real), len(feat_fake))
    similarities = []
    for i in range(N):
        norm_real = np.linalg.norm(feat_real[i])
        norm_fake = np.linalg.norm(feat_fake[i])
        if norm_real > 1e-8 and norm_fake > 1e-8:
            sim = np.dot(feat_real[i], feat_fake[i]) / (norm_real * norm_fake)
            similarities.append(sim)
    
    return float(np.mean(similarities)) if similarities else 0.0


def compute_ajsd(real_data, fake_data, n_bins=50):
    """
    Compute Average Jensen-Shannon Distance (AJSD) between real and fake data.
    Score in [0, 1], closer to 0 is better.
    """
    from scipy.spatial.distance import jensenshannon
    
    feat_real = extract_statistical_features(real_data)
    feat_fake = extract_statistical_features(fake_data)
    
    D = feat_real.shape[1]
    jsd_values = []
    
    for d in range(D):
        # Get min/max across both for consistent binning
        all_vals = np.concatenate([feat_real[:, d], feat_fake[:, d]])
        min_val, max_val = np.min(all_vals), np.max(all_vals)
        
        if max_val - min_val < 1e-8:
            continue  # Skip constant features
        
        bins = np.linspace(min_val, max_val, n_bins + 1)
        
        # Compute histograms (probability distributions)
        hist_real, _ = np.histogram(feat_real[:, d], bins=bins, density=True)
        hist_fake, _ = np.histogram(feat_fake[:, d], bins=bins, density=True)
        
        # Add small epsilon to avoid division by zero
        hist_real = hist_real + 1e-10
        hist_fake = hist_fake + 1e-10
        
        # Normalize to get proper probability distributions
        hist_real = hist_real / hist_real.sum()
        hist_fake = hist_fake / hist_fake.sum()
        
        # Jensen-Shannon distance (sqrt of JS divergence)
        jsd = jensenshannon(hist_real, hist_fake)
        jsd_values.append(jsd)
    
    return float(np.mean(jsd_values)) if jsd_values else 0.0


# ============================================================================
# Discriminative and Predictive Scores (TimeGAN protocol)
# ============================================================================

class LSTMClassifier(nn.Module):
    """2-layer LSTM classifier for discriminative score."""
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=2, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
    
    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        out = self.fc(h_n[-1])
        return torch.sigmoid(out)


class LSTMPredictor(nn.Module):
    """2-layer LSTM predictor for predictive score."""
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=2, batch_first=True)
        self.fc = nn.Linear(hidden_dim, input_dim)
    
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out)


def compute_discriminative_score(real_data, fake_data, device='cpu', epochs=500, batch_size=256):
    """
    Compute Discriminative Score using a 2-layer LSTM classifier.
    Score = |accuracy - 0.5|, closer to 0 is better.
    """
    from torch.utils.data import TensorDataset, DataLoader
    
    # Prepare data: real=1, fake=0
    real_tensor = torch.tensor(real_data, dtype=torch.float32)
    fake_tensor = torch.tensor(fake_data, dtype=torch.float32)
    
    real_labels = torch.ones(len(real_data), 1)
    fake_labels = torch.zeros(len(fake_data), 1)
    
    # Combine and shuffle
    all_data = torch.cat([real_tensor, fake_tensor], dim=0)
    all_labels = torch.cat([real_labels, fake_labels], dim=0)
    
    # Train/test split (80/20)
    n_samples = len(all_data)
    indices = torch.randperm(n_samples)
    train_size = int(0.8 * n_samples)
    
    train_data = all_data[indices[:train_size]]
    train_labels = all_labels[indices[:train_size]]
    test_data = all_data[indices[train_size:]]
    test_labels = all_labels[indices[train_size:]]
    
    train_loader = DataLoader(TensorDataset(train_data, train_labels), batch_size=batch_size, shuffle=True)
    
    # Initialize model
    input_dim = real_data.shape[2]
    model = LSTMClassifier(input_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()
    
    # Train
    model.train()
    for _ in range(epochs):
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            pred = model(batch_x)
            loss = criterion(pred, batch_y)
            loss.backward()
            optimizer.step()
    
    # Evaluate
    model.eval()
    with torch.no_grad():
        test_data = test_data.to(device)
        test_labels = test_labels.to(device)
        pred = model(test_data)
        pred_labels = (pred > 0.5).float()
        accuracy = (pred_labels == test_labels).float().mean().item()
    
    discriminative_score = abs(accuracy - 0.5)
    return discriminative_score


def compute_predictive_score(real_data, fake_data, device='cpu', epochs=500, batch_size=256):
    """
    Compute Predictive Score using TSTR paradigm with a 2-layer LSTM.
    Train on synthetic, test on real. Returns MAE, lower is better.
    """
    from torch.utils.data import TensorDataset, DataLoader
    
    # Prepare training data from fake samples
    # Input: x_{1:T-1}, Target: x_{2:T}
    fake_tensor = torch.tensor(fake_data, dtype=torch.float32)
    fake_input = fake_tensor[:, :-1, :]
    fake_target = fake_tensor[:, 1:, :]
    
    train_loader = DataLoader(TensorDataset(fake_input, fake_target), batch_size=batch_size, shuffle=True)
    
    # Initialize model
    input_dim = real_data.shape[2]
    model = LSTMPredictor(input_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.L1Loss()
    
    # Train on fake data
    model.train()
    for _ in range(epochs):
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            pred = model(batch_x)
            loss = criterion(pred, batch_y)
            loss.backward()
            optimizer.step()
    
    # Test on real data
    real_tensor = torch.tensor(real_data, dtype=torch.float32)
    real_input = real_tensor[:, :-1, :]
    real_target = real_tensor[:, 1:, :]
    
    model.eval()
    with torch.no_grad():
        real_input = real_input.to(device)
        real_target = real_target.to(device)
        pred = model(real_input)
        mae = torch.mean(torch.abs(pred - real_target)).item()
    
    return mae


def t_sne(features_fake, features_real):
    """Compute t-SNE embeddings."""
    from sklearn.manifold import TSNE
    dist = np.concatenate((features_fake, features_real), axis=0)
    tsne = TSNE(n_components=2, random_state=42)
    res = tsne.fit_transform(dist)
    res_fake = res[:len(features_fake)]
    res_real = res[len(features_fake):]
    return res_fake, res_real


def pca(features_fake, features_real):
    """Compute PCA embeddings."""
    from sklearn.decomposition import PCA
    pca_model = PCA(n_components=2)
    res_real = pca_model.fit_transform(features_real)
    res_fake = pca_model.transform(features_fake)
    return res_fake, res_real


def evaluate_model(loss_name, generator, real_dataset, config, output_dir, num_runs=10):
    """Evaluate a trained model and save results with mean and std over multiple runs."""
    print(f"\n  Evaluating {loss_name.upper()} over {num_runs} runs...")
    
    device = config['device']
    num_samples = config['num_eval_samples']
    seq_len = config['seq_len']
    channels = config['channels']
    # Get real samples (N, seq_len, channels)
    real_samples = real_dataset[:num_samples].squeeze().reshape(num_samples, seq_len, channels).numpy()

    # Storage for metrics across runs
    all_mmd = []
    all_fd = []
    all_acs = []
    all_ajsd = []
    all_disc = []
    all_pred = []

    for run_idx in range(num_runs):
        print(f"\n    Run {run_idx + 1}/{num_runs}...")
        
        # Generate NEW fake samples for each run
        fake_samples = generate_samples(generator, num_samples, config['latent_dim'], device, seq_len, channels)

        # Extract ROCKET features
        features_fake, features_real = feature_extract(fake_samples, real_samples)

        # ============ Compute all metrics ============
        print("      Computing MMD...")
        mmd_score = compute_mmd(features_fake, features_real, device)
        all_mmd.append(mmd_score)
        
        print("      Computing Frechet Distance...")
        fd_score = compute_frechet_distance(features_fake, features_real, device)
        all_fd.append(fd_score)
        
        print("      Computing ACS...")
        acs_score = compute_acs(real_samples, fake_samples)
        all_acs.append(acs_score)
        
        print("      Computing AJSD...")
        ajsd_score = compute_ajsd(real_samples, fake_samples)
        all_ajsd.append(ajsd_score)
        
        print("      Computing Discriminative Score...")
        disc_score = compute_discriminative_score(real_samples, fake_samples, device)
        all_disc.append(disc_score)
        
        print("      Computing Predictive Score...")
        pred_score = compute_predictive_score(real_samples, fake_samples, device)
        all_pred.append(pred_score)
        
        print(f"      Run {run_idx + 1} results: MMD={mmd_score:.6f}, FD={fd_score:.2f}, ACS={acs_score:.4f}, AJSD={ajsd_score:.4f}, Disc={disc_score:.4f}, Pred={pred_score:.4f}")

    # Compute mean and std for each metric
    metrics_stats = {
        "MMD": (np.mean(all_mmd), np.std(all_mmd)),
        "FD": (np.mean(all_fd), np.std(all_fd)),
        "ACS": (np.mean(all_acs), np.std(all_acs)),
        "AJSD": (np.mean(all_ajsd), np.std(all_ajsd)),
        "Discriminative": (np.mean(all_disc), np.std(all_disc)),
        "Predictive": (np.mean(all_pred), np.std(all_pred)),
    }

    print(f"\n    Summary for {loss_name.upper()} over {num_runs} runs:")
    for metric_name, (mean_val, std_val) in metrics_stats.items():
        print(f"      {metric_name}: {mean_val:.6f} ± {std_val:.6f}")

    # ============ Visualizations (using last run's samples) ============
    # t-SNE and PCA
    tsne_fake, tsne_real = t_sne(features_fake, features_real)
    pca_fake, pca_real = pca(features_fake, features_real)

    # Plot t-SNE
    plt.figure(figsize=(10, 7))
    plt.scatter(tsne_real[:, 0], tsne_real[:, 1], color='blue', alpha=0.5, label='Real')
    plt.scatter(tsne_fake[:, 0], tsne_fake[:, 1], color='red', alpha=0.5, label='Generated')
    plt.title(f"t-SNE Visualization - {loss_name.upper()}")
    plt.legend()
    plt.show()

    # Plot PCA
    plt.figure(figsize=(10, 7))
    plt.scatter(pca_real[:, 0], pca_real[:, 1], color='blue', alpha=0.5, label='Real')
    plt.scatter(pca_fake[:, 0], pca_fake[:, 1], color='red', alpha=0.5, label='Generated')
    plt.title(f"PCA Visualization - {loss_name.upper()}")
    plt.legend()
    plt.savefig(os.path.join(output_dir, f"{loss_name}_pca.png"), dpi=150)
    plt.show()

    # Plot generated samples
    generator.eval()
    with torch.no_grad():
        noise = torch.randn(8, config['latent_dim'], device=device)
        gen_samples = generator(noise).cpu().numpy()

    real_batch = real_dataset[:8]
    fig, axs = plt.subplots(4, 2, figsize=(15, 10))
    fig.suptitle(f"Real vs Generated - {loss_name.upper()}", fontsize=16)
    axs = axs.flatten()
    for i in range(8):
        if i % 2 == 0:
            axs[i].plot(real_batch[i].squeeze().numpy().T, color='blue', alpha=0.7)
            axs[i].set_title("Real")
        else:
            axs[i].plot(gen_samples[i].squeeze().T, color='red', alpha=0.7)
            axs[i].set_title("Generated")
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(os.path.join(output_dir, f"{loss_name}_gen.png"), dpi=150)
    plt.show()

    return {
        "loss_name": loss_name,
        "MMD": metrics_stats["MMD"],
        "FD": metrics_stats["FD"],
        "ACS": metrics_stats["ACS"],
        "AJSD": metrics_stats["AJSD"],
        "Discriminative": metrics_stats["Discriminative"],
        "Predictive": metrics_stats["Predictive"],
        "num_runs": num_runs
    }

In [ ]:
# ============================================================================
# MAIN EXECUTION
# ============================================================================

# Results storage
all_metrics = []
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

for loss_name in LOSSES_TO_RUN:
    # Prepare config for this specific loss with overrides
    current_config = TRAIN_CONFIG.copy()
    if loss_name in LOSS_CONFIG_OVERRIDES:
        print(f"Applying config overrides for {loss_name}: {LOSS_CONFIG_OVERRIDES[loss_name]}")
        current_config.update(LOSS_CONFIG_OVERRIDES[loss_name])
        
    loss_params = LOSS_HYPERPARAMS.get(loss_name, {}).copy()
    
    if not EVAL_ONLY:
        generator = train_single_loss(loss_name, current_config, loss_params, dataloader, current_config['output_dir'])
    else:
        # Load existing model
        ckpt_path = os.path.join(current_config['output_dir'], f"ckpt_{loss_name}", "generator_final.pth")
        if not os.path.exists(ckpt_path):
            print(f"Warning: No trained model found for {loss_name}, skipping...")
            continue
        generator = Generator(
            seq_len=current_config['seq_len'],
            patch_size=current_config['patch_size'],
            channels=current_config['channels'],
            latent_dim=current_config['latent_dim'],
            depth=5
        ).to(current_config['device'])
        generator.load_state_dict(torch.load(ckpt_path, map_location=current_config['device']))

    # Evaluate (runs 10 times by default)
    metrics = evaluate_model(loss_name, generator, dataset, current_config, current_config['output_dir'], num_runs=10)
    all_metrics.append(metrics)

# Print Summary
print("\n" + "=" * 120)
print("SUMMARY (Mean ± Std over 10 runs)")
print("=" * 120)
print(f"{'Loss':<12} | {'MMD':>20} | {'FD':>20} | {'ACS':>16} | {'AJSD':>16} | {'Disc':>16} | {'Pred':>16}")
print("-" * 120)
for m in all_metrics:
    mmd_mean, mmd_std = m['MMD']
    fd_mean, fd_std = m['FD']
    acs_mean, acs_std = m['ACS']
    ajsd_mean, ajsd_std = m['AJSD']
    disc_mean, disc_std = m['Discriminative']
    pred_mean, pred_std = m['Predictive']
    print(f"{m['loss_name'].upper():<12} | {mmd_mean:>9.6f}±{mmd_std:<9.6f} | {fd_mean:>9.2f}±{fd_std:<9.2f} | {acs_mean:>7.4f}±{acs_std:<7.4f} | {ajsd_mean:>7.4f}±{ajsd_std:<7.4f} | {disc_mean:>7.4f}±{disc_std:<7.4f} | {pred_mean:>7.4f}±{pred_std:<7.4f}")
print("=" * 120)

# Save metrics to file
metrics_file = os.path.join(TRAIN_CONFIG['output_dir'], f"metrics_{timestamp}.txt")
with open(metrics_file, "w") as f:
    f.write(f"Evaluation Results - {timestamp}\n")
    f.write("=" * 120 + "\n\n")
    f.write("Metric Descriptions:\n")
    f.write("  MMD: Maximum Mean Discrepancy (lower is better)\n")
    f.write("  FD: Frechet Distance using ROCKET features (lower is better)\n")
    f.write("  ACS: Average Cosine Similarity (higher is better, max 1.0)\n")
    f.write("  AJSD: Average Jensen-Shannon Distance (lower is better, min 0.0)\n")
    f.write("  Disc: Discriminative Score (lower is better, min 0.0)\n")
    f.write("  Pred: Predictive Score/MAE (lower is better)\n\n")
    f.write("All metrics are reported as Mean ± Std over 10 runs\n\n")
    f.write("-" * 120 + "\n")
    f.write(f"{'Loss':<12} | {'MMD':>20} | {'FD':>20} | {'ACS':>16} | {'AJSD':>16} | {'Disc':>16} | {'Pred':>16}\n")
    f.write("-" * 120 + "\n")
    for m in all_metrics:
        mmd_mean, mmd_std = m['MMD']
        fd_mean, fd_std = m['FD']
        acs_mean, acs_std = m['ACS']
        ajsd_mean, ajsd_std = m['AJSD']
        disc_mean, disc_std = m['Discriminative']
        pred_mean, pred_std = m['Predictive']
        f.write(f"{m['loss_name'].upper():<12} | {mmd_mean:>9.6f}±{mmd_std:<9.6f} | {fd_mean:>9.2f}±{fd_std:<9.2f} | {acs_mean:>7.4f}±{acs_std:<7.4f} | {ajsd_mean:>7.4f}±{ajsd_std:<7.4f} | {disc_mean:>7.4f}±{disc_std:<7.4f} | {pred_mean:>7.4f}±{pred_std:<7.4f}\n")
    f.write("=" * 120 + "\n")
print(f"\nMetrics saved to: {metrics_file}")